# 00. Верификация схемы Hive-таблиц

**Цель**: Проверить предполагаемые имена колонок в `src/schema.py` против реальной схемы таблиц на Hadoop.

**ВАЖНО**: Этот ноутбук должен быть запущен ПЕРВЫМ на MDP, до выполнения любых других ноутбуков.

---

In [ ]:
import sys
sys.path.insert(0, '..')

from pyspark.sql import SparkSession
from src import schema
from src.config import HIVE_DATABASE

In [ ]:
spark = SparkSession.builder \
    .enableHiveSupport() \
    .getOrCreate()

print(f"SparkSession initialized. Hive database: {HIVE_DATABASE}")
spark.sql(f"USE {HIVE_DATABASE}")
spark.sql("SHOW TABLES").show(20, truncate=False)

## Проверка всех ключевых таблиц

Для каждой таблицы:
1. Выполняем `DESCRIBE TABLE`
2. Сравниваем реальные колонки с предполагаемыми в `schema.py`
3. Выводим несовпадения

In [ ]:
def verify_table(spark, table_name, assumed_columns):
    """Compare assumed column names with actual Hive schema."""
    full_name = f"{HIVE_DATABASE}.{table_name}"
    print(f"\n{'='*60}")
    print(f"Table: {full_name}")
    print('='*60)
    
    try:
        desc_df = spark.sql(f"DESCRIBE {full_name}")
        actual_cols = [row['col_name'] for row in desc_df.collect()
                       if not row['col_name'].startswith('#')]
        
        assumed_set = set(assumed_columns.values())
        actual_set = set(actual_cols)
        
        matched = assumed_set & actual_set
        missing = assumed_set - actual_set  # assumed but not in table
        extra = actual_set - assumed_set    # in table but not assumed
        
        print(f"  Actual columns: {len(actual_cols)}")
        print(f"  Assumed columns: {len(assumed_set)}")
        print(f"  Matched: {len(matched)}")
        
        if missing:
            print(f"  MISSING (assumed but not found): {sorted(missing)}")
        if extra:
            print(f"  Extra (in table, not assumed): {sorted(extra)[:20]}")
            if len(extra) > 20:
                print(f"    ...and {len(extra)-20} more")
        
        if not missing:
            print("  STATUS: OK")
        else:
            print("  STATUS: NEEDS FIX")
        
        return {
            'table': table_name,
            'actual_cols': actual_cols,
            'matched': matched,
            'missing': missing,
            'extra': extra,
            'ok': len(missing) == 0
        }
        
    except Exception as e:
        print(f"  ERROR: {e}")
        return {'table': table_name, 'error': str(e), 'ok': False}

In [ ]:
results = {}
for table_name, columns in schema.ALL_TABLES.items():
    results[table_name] = verify_table(spark, table_name, columns)

## Сводка результатов

In [ ]:
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

all_ok = True
for table_name, result in results.items():
    status = "OK" if result.get('ok') else "NEEDS FIX"
    error = result.get('error', '')
    if error:
        status = f"ERROR: {error[:50]}"
    print(f"  {table_name:45s} {status}")
    if not result.get('ok'):
        all_ok = False

if all_ok:
    print("\nAll tables verified. schema.py is correct.")
else:
    print("\nSome tables need fixes. See corrections below.")

## Автоматическая генерация исправлений

Если есть несовпадения, ячейка ниже генерирует исправленный фрагмент `schema.py`.

In [ ]:
for table_name, result in results.items():
    if result.get('error') or result.get('ok'):
        continue
    
    missing = result.get('missing', set())
    extra = result.get('extra', set())
    actual_cols = result.get('actual_cols', [])
    
    print(f"\n# --- Fix for {table_name} ---")
    print(f"# Missing assumed columns: {sorted(missing)}")
    print(f"# Available columns not in schema:")
    for col in sorted(extra):
        print(f"#   '{col}'")
    print(f"\n# Full column list:")
    for col in actual_cols:
        print(f"#   '{col}'")

## Просмотр примеров данных (для проверки содержимого)

In [ ]:
# Показать первые строки ключевых таблиц
KEY_TABLES = [
    'paymentcounteragent_stran',  # Главная — транзакции
    'clientauthority_shist',       # Доверенности
    'clientauthority2clientrb_shist',  # Связи
    'clnt2dealsalary_shist',       # Зарплатные проекты
    'dealsalary_sdim',             # Зарплатные сделки
]

for tbl in KEY_TABLES:
    print(f"\n{'='*60}")
    print(f"Sample: {HIVE_DATABASE}.{tbl}")
    print('='*60)
    try:
        if tbl == 'paymentcounteragent_stran':
            # Partitioned table — must filter
            spark.sql(f"""
                SELECT * FROM {HIVE_DATABASE}.{tbl}
                WHERE date_part >= '2025-10-01' AND date_part <= '2025-10-31'
                LIMIT 5
            """).show(truncate=False)
        else:
            spark.sql(f"SELECT * FROM {HIVE_DATABASE}.{tbl} LIMIT 5").show(truncate=False)
    except Exception as e:
        print(f"  ERROR: {e}")